# Fashion VLM — inference playground (Qwen2-VL-2B + LoRA)
Loads the LoRA adapter trained in *Vision-Language Model (VLM) fine-tuning for VQA* and tests it on images from the Kaggle input or from a URL.

| Part | Content |
|---|---|
| 0 | Installation, locating the inputs, loading base model + adapter |
| 1 | Inference toolkit: `ask`, `product_sheet`, `top_k`, `audit_listing`, `compare` |
| 2 | Demos on a validation image and on an image from a URL |
| 3 | Batch auto-tagging of catalog images vs. `styles.csv` ground truth |
| 4 | Publish the adapter to the Hugging Face Hub (for the Space) |

**Inputs (Add Input):**
- The Kaggle Model with the adapter (`adapter_best/` files + `training_config.json` + `summary_vlm.json` + `eval_predictions.csv`), or the output of the training notebook.
- *Optional:* `paramaggarwal/fashion-product-images-dataset` for Parts 2–3.

**Accelerator:** GPU T4 (one GPU is enough). Internet ON (downloads `Qwen/Qwen2-VL-2B-Instruct`).

# Part 0 — Setup

In [ ]:
!pip install -q -U "transformers>=4.49,<5" "accelerate>=1.0" "peft>=0.13"
!pip uninstall -y -q torchao

In [ ]:
import os, io, re, json, random, shutil
from pathlib import Path
from contextlib import nullcontext

import requests
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from IPython.display import display
from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoVLM
from peft import PeftModel

ImageFile.LOAD_TRUNCATED_IMAGES = True
plt.rcParams["figure.dpi"] = 110

## Part 0.1 — Locate the inputs
The paths depend on how the input was attached (Kaggle Model, dataset or notebook output), so the files are searched by suffix.

In [ ]:
INPUT_ROOT = "/kaggle/input"
INPUT_FILES = sorted(os.path.join(d, f) for d, _, fs in os.walk(INPUT_ROOT) for f in fs
                     if not f.lower().endswith((".jpg", ".jpeg", ".png")))


def find_input(suffix):
    return next((p for p in INPUT_FILES if p.endswith(suffix)), None)


adapter_cfg = find_input("/adapter_best/adapter_config.json") or find_input("/adapter_config.json")
assert adapter_cfg, "adapter not found: add the Kaggle Model (or the training notebook output) as input"
ADAPTER_DIR = os.path.dirname(adapter_cfg)
TRAIN_CFG = json.load(open(find_input("/training_config.json") or find_input("/vlm/config.json")))
VOCAB = TRAIN_CFG["vocab"]
SUMMARY_PATH = find_input("/summary_vlm.json")
EVAL_PRED_PATH = find_input("/eval_predictions.csv")
STYLES_CSV = find_input("/styles.csv")
IMG_DIR = os.path.join(os.path.dirname(STYLES_CSV), "images") if STYLES_CSV else None

for k, v in {"adapter": ADAPTER_DIR, "base model": TRAIN_CFG["model_name"], "summary": SUMMARY_PATH,
             "eval predictions": EVAL_PRED_PATH, "styles.csv": STYLES_CSV, "images": IMG_DIR}.items():
    print(f"{k:17s} {v}")

## Part 0.2 — Base model + LoRA adapter
Same setup as the evaluation of the training notebook: fp16 weights, SDPA attention, greedy decoding and the same visual-token budget (`image_min_pixels` / `image_max_pixels`).

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

base = AutoVLM.from_pretrained(TRAIN_CFG["model_name"], dtype=DTYPE, attn_implementation="sdpa").to(DEVICE)
base.generation_config.do_sample = False
for k in ("temperature", "top_p", "top_k"):
    setattr(base.generation_config, k, None)
model = PeftModel.from_pretrained(base, ADAPTER_DIR).eval()

processor = AutoProcessor.from_pretrained(ADAPTER_DIR, min_pixels=TRAIN_CFG["image_min_pixels"],
                                          max_pixels=TRAIN_CFG["image_max_pixels"])
processor.tokenizer.padding_side = "left"
if DEVICE == "cuda":
    print(f"GPU memory: {torch.cuda.memory_allocated() / 2**30:.1f} GB")

# Part 1 — Inference toolkit
- `ask(image, question)`: free question (short catalog answers; `compare=True` adds the zero-shot answer).
- `product_sheet(image)`: the 7 catalog attributes + description in one batched call, plus a catalog title and a JSON record.
- `top_k(image, question, k)`: beam search returns the *k* most likely answers with their probability (sum of token log-probs, `length_penalty=0`).
- `audit_listing(image, claimed)`: checks a product listing against the image (predicted value + yes/no confirmation for color and type).
- `compare(image, questions)`: zero-shot vs fine-tuned with the same weights (adapter on/off).

`image` can be a `PIL.Image`, a local path (e.g. under `/kaggle/input`) or an `http(s)` URL.

In [ ]:
SYSTEM_PROMPT = ("You are a fashion catalog assistant. Look carefully at the product image and "
                 "answer the question with a short, exact answer using the catalog vocabulary.")
ATTR_QUESTIONS = {
    "baseColour": "What color is this product?",
    "articleType": "What type of product is this?",
    "subCategory": "Which subcategory does this product belong to?",
    "masterCategory": "What is the master category of this product?",
    "gender": "Who is this product for?",
    "usage": "What is this product used for?",
    "season": "Which season is this product for?",
}
DESCRIBE_QUESTION = "Describe this product."
GENDER_POSSESSIVE = {"men": "men's", "women": "women's", "boys": "boys'", "girls": "girls'", "unisex": "unisex"}
SHORT_TOKENS, LONG_TOKENS = 16, 60

def load_any(src):
    if isinstance(src, Image.Image):
        return src.convert("RGB")
    if str(src).startswith(("http://", "https://")):
        r = requests.get(src, timeout=20, headers={"User-Agent": "Mozilla/5.0 (fashion-vlm-demo)"})
        r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert("RGB")
    return Image.open(src).convert("RGB")

def prompt(question):
    msgs = [{"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def encode(images, questions):
    return processor(text=[prompt(q) for q in questions], images=images, padding=True,
                     return_tensors="pt").to(DEVICE)

def run_ctx(use_adapter):
    ctx = [nullcontext() if use_adapter else model.disable_adapter()]
    ctx.append(torch.autocast("cuda", dtype=torch.float16) if DEVICE == "cuda" else nullcontext())
    return ctx

@torch.no_grad()
def generate(images, questions, use_adapter=True, max_new_tokens=SHORT_TOKENS):
    enc = encode(images, questions)
    adapter_ctx, amp_ctx = run_ctx(use_adapter)
    with adapter_ctx, amp_ctx:
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=processor.tokenizer.pad_token_id)
    return [s.strip() for s in processor.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]

def tokens_for(question):
    return LONG_TOKENS if re.search(r"describ|tell me about", question, re.I) else SHORT_TOKENS

def norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s\-']", " ", str(s).lower())).strip()

In [ ]:
def ask(image, question, compare=False, max_new_tokens=None):
    img = load_any(image)
    mnt = max_new_tokens or tokens_for(question)
    out = {"question": question, "fine_tuned": generate([img], [question], True, mnt)[0]}
    if compare:
        out["zero_shot"] = generate([img], [question], False, mnt)[0]
    return out

def catalog_title(sheet):
    words = [GENDER_POSSESSIVE.get(sheet.get("gender", ""), ""), sheet.get("baseColour", ""), sheet.get("articleType", "")]
    return " ".join(w for w in words if w).title()

def product_sheet(image, describe=True, use_adapter=True):
    img = load_any(image)
    questions = list(ATTR_QUESTIONS.values()) + ([DESCRIBE_QUESTION] if describe else [])
    answers = generate([img] * len(questions), questions, use_adapter, LONG_TOKENS if describe else SHORT_TOKENS)
    sheet = dict(zip(list(ATTR_QUESTIONS) + (["description"] if describe else []), answers))
    sheet["title"] = catalog_title(sheet)
    return sheet

@torch.no_grad()
def top_k(image, question, k=5, use_adapter=True):
    enc = encode([load_any(image)], [question])
    adapter_ctx, amp_ctx = run_ctx(use_adapter)
    with adapter_ctx, amp_ctx:
        out = model.generate(**enc, max_new_tokens=tokens_for(question), num_beams=k, num_return_sequences=k,
                             do_sample=False, length_penalty=0.0, early_stopping=True, output_scores=True,
                             return_dict_in_generate=True, pad_token_id=processor.tokenizer.pad_token_id)
    answers = processor.batch_decode(out.sequences[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    probs = {}
    for a, s in zip(answers, out.sequences_scores.float().exp().tolist()):
        probs[a.strip()] = probs.get(a.strip(), 0.0) + s
    return dict(sorted(probs.items(), key=lambda x: -x[1]))

YESNO_CHECKS = {"baseColour": "Is this item {v}?", "articleType": "Is this a {v} product?"}

def audit_listing(image, claimed):
    img = load_any(image)
    sheet = product_sheet(img, describe=False)
    checks = {a: YESNO_CHECKS[a].format(v=v.lower()) for a, v in claimed.items() if a in YESNO_CHECKS}
    confirm = dict(zip(checks, generate([img] * len(checks), list(checks.values())))) if checks else {}
    rows = []
    for attr, value in claimed.items():
        match = norm(value) == norm(sheet.get(attr, ""))
        yn = confirm.get(attr, "")
        rows.append({"attribute": attr, "claimed": value.lower(), "predicted": sheet.get(attr, ""),
                     "match": match, "yes/no check": yn,
                     "verdict": "OK" if match or norm(yn) == "yes" else "REVIEW"})
    return pd.DataFrame(rows)

def compare(image, questions):
    img = load_any(image)
    rows = []
    for q in questions:
        mnt = tokens_for(q)
        rows.append({"question": q, "zero_shot": generate([img], [q], False, mnt)[0],
                     "fine_tuned": generate([img], [q], True, mnt)[0]})
    return pd.DataFrame(rows)

def show(image, title="", width=3):
    plt.figure(figsize=(width, width * 1.3))
    plt.imshow(load_any(image)); plt.axis("off"); plt.title(title, fontsize=8); plt.show()

# Part 2 — Demos
## Part 2.1 — A validation image (never seen in training)
The image is taken from the evaluation set of the training notebook (`eval_predictions.csv`), so the ground truth from `styles.csv` can be shown next to the predictions.

In [ ]:
styles = None
if STYLES_CSV:
    styles = pd.read_csv(STYLES_CSV, on_bad_lines="skip")
    styles.columns = [c.strip() for c in styles.columns]
    styles = styles.set_index("id")

eval_ids = pd.read_csv(EVAL_PRED_PATH)["id"].unique().tolist() if EVAL_PRED_PATH else []
pool = [i for i in eval_ids if styles is not None and i in styles.index and os.path.exists(f"{IMG_DIR}/{i}.jpg")]
if not pool and IMG_DIR:
    pool = [int(f.split(".")[0]) for f in sorted(os.listdir(IMG_DIR))[:500]]

random.seed(7)
DEMO_ID = random.choice(pool) if pool else None
DEMO_IMAGE = load_any(f"{IMG_DIR}/{DEMO_ID}.jpg") if DEMO_ID else None
if DEMO_IMAGE is not None:
    truth = styles.loc[DEMO_ID] if styles is not None and DEMO_ID in styles.index else None
    show(DEMO_IMAGE, truth["productDisplayName"] if truth is not None else str(DEMO_ID))
    sheet = product_sheet(DEMO_IMAGE)
    table = pd.DataFrame({"prediction": pd.Series(sheet)})
    if truth is not None:
        table["ground truth"] = [str(truth[a]).lower() if a in truth.index else "" for a in table.index]
    display(table)
    print(json.dumps(sheet, indent=2))

## Part 2.2 — Free questions, top-k confidence and zero-shot vs fine-tuned

In [ ]:
if DEMO_IMAGE is not None:
    for q in ["Is this item black?", "Is this item for men, women, boys, girls or unisex?",
              "What kind of article is this?", "Give a short description of this item."]:
        print(ask(DEMO_IMAGE, q))

In [ ]:
if DEMO_IMAGE is not None:
    for q in ["What color is this product?", "What type of product is this?", "Which season is this product for?"]:
        probs = top_k(DEMO_IMAGE, q, k=5)
        print(q)
        for a, p in probs.items():
            print(f"   {p:6.1%}  {a}")

In [ ]:
if DEMO_IMAGE is not None:
    display(compare(DEMO_IMAGE, ["What color is this product?", "What type of product is this?",
                                 "Who is this product for?", "Describe this product."]))

## Part 2.3 — Listing audit
Simulates a seller listing with a wrong color: the audit flags the attributes that disagree with the image.

In [ ]:
if DEMO_IMAGE is not None and truth is not None:
    wrong_colour = next(c for c in VOCAB["baseColour"] if not set(c.split()) & set(str(truth["baseColour"]).lower().split()))
    claimed = {"baseColour": wrong_colour, "articleType": str(truth["articleType"]),
               "gender": str(truth["gender"]), "usage": str(truth["usage"])}
    display(audit_listing(DEMO_IMAGE, claimed))

## Part 2.4 — Image from a URL
Any public image URL works. The model was trained on catalog packshots (single product, light background), so it is most reliable on that kind of picture.

In [ ]:
IMAGE_URL = "https://unsplash.com/photos/KPydzVyWfqg/download?w=640"

try:
    url_image = load_any(IMAGE_URL)
    show(url_image, "image from URL")
    display(pd.DataFrame({"prediction": pd.Series(product_sheet(url_image))}))
    print(top_k(url_image, "What color is this product?", k=3))
except Exception as e:
    print("could not process the URL:", e)

# Part 3 — Batch auto-tagging
Tags a list of images (one batched generation per attribute) and, when the images come from the catalog, measures exact-match accuracy against `styles.csv`. `tag_images` accepts any list of paths, e.g. a folder of your own images under `/kaggle/input`.

In [ ]:
def tag_images(paths, batch_size=16, describe=False):
    rows = [{"image": p} for p in paths]
    questions = dict(ATTR_QUESTIONS, **({"description": DESCRIBE_QUESTION} if describe else {}))
    for s in range(0, len(paths), batch_size):
        imgs = [load_any(p) for p in paths[s:s + batch_size]]
        for attr, q in questions.items():
            preds = generate(imgs, [q] * len(imgs), max_new_tokens=tokens_for(q))
            for r, p in zip(rows[s:s + batch_size], preds):
                r[attr] = p
    return pd.DataFrame(rows)


N_BATCH = 48
if pool:
    batch_ids = pool[:N_BATCH]
    tags = tag_images([f"{IMG_DIR}/{i}.jpg" for i in batch_ids])
    tags.insert(0, "id", batch_ids)
    if styles is not None:
        acc = {a: (tags[a].map(norm) == styles.loc[batch_ids, a].astype(str).map(norm).values).mean() for a in ATTR_QUESTIONS}
        display(pd.Series(acc, name="exact match").to_frame().style.format("{:.1%}"))
    tags.to_csv("/kaggle/working/batch_tags.csv", index=False)
    display(tags.head(10))

In [ ]:
if pool:
    fig, axes = plt.subplots(1, 6, figsize=(14, 3.4))
    for ax, (_, r) in zip(axes, tags.head(6).iterrows()):
        ax.imshow(load_any(r["image"])); ax.axis("off")
        ax.set_title(catalog_title(r.to_dict()), fontsize=7)
    plt.tight_layout(); plt.show()

# Part 4 — Publish the adapter to the Hugging Face Hub
Builds `/kaggle/working/hf_model/` (adapter + processor + catalog vocabulary + metrics + example images + model card) and uploads it as a **model repo**. The Space downloads it at start-up.

1. Create a token with *write* permission at huggingface.co/settings/tokens.
2. In Kaggle: *Add-ons → Secrets* → name `HF_TOKEN`, attach it to this notebook.
3. Set `PUBLISH = True` and run.

In [ ]:
PUBLISH = False
REPO_NAME = "qwen2-vl-2b-fashion-vqa-lora"
STAGE = Path("/kaggle/working/hf_model")

shutil.rmtree(STAGE, ignore_errors=True)
shutil.copytree(ADAPTER_DIR, STAGE, ignore=shutil.ignore_patterns("README.md", "*metadata.json", "training_config.json", "eval_predictions.csv", "summary_vlm.json"))
json.dump(VOCAB, open(STAGE / "catalog_vocab.json", "w"), indent=1)
summary = json.load(open(SUMMARY_PATH)) if SUMMARY_PATH else {}
if summary:
    json.dump(summary, open(STAGE / "summary_vlm.json", "w"), indent=2)

(STAGE / "examples").mkdir()
for i in pool[:8]:
    load_any(f"{IMG_DIR}/{i}.jpg").resize((300, 400)).save(STAGE / "examples" / f"{i}.jpg", quality=90)

metrics = "".join(f"| {r['qtype']} | {r['base_exact']:.3f} | {r['ft_exact']:.3f} |\n" for r in summary.get("by_qtype", []))
card = f"""---
base_model: {TRAIN_CFG['model_name']}
library_name: peft
pipeline_tag: image-text-to-text
license: apache-2.0
tags: [lora, qwen2-vl, vqa, fashion]
---
# Qwen2-VL-2B fashion VQA (LoRA)
LoRA adapter (r={TRAIN_CFG['lora_r']}, alpha={TRAIN_CFG['lora_alpha']}, LLM linear layers) for `{TRAIN_CFG['model_name']}`, fine-tuned on the
Fashion Product Images dataset to answer catalog questions (color, article type, category, gender, usage, season,
yes/no checks) and write short product descriptions.

Overall exact match on the validation set: zero-shot **{summary.get('overall', {}).get('base_exact', 0):.3f}** → fine-tuned **{summary.get('overall', {}).get('ft_exact', 0):.3f}**.

| question type | zero-shot | fine-tuned |
|---|---|---|
{metrics}
## Usage
```python
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
base = AutoModelForImageTextToText.from_pretrained("{TRAIN_CFG['model_name']}", dtype="auto", device_map="auto")
model = PeftModel.from_pretrained(base, "<this repo>")
processor = AutoProcessor.from_pretrained("<this repo>", min_pixels={TRAIN_CFG['image_min_pixels']}, max_pixels={TRAIN_CFG['image_max_pixels']})
```
System prompt used in training: *"{SYSTEM_PROMPT}"*
"""
(STAGE / "README.md").write_text(card)
print(sorted(p.name for p in STAGE.iterdir()))

if PUBLISH:
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    repo_id = f"{api.whoami()['name']}/{REPO_NAME}"
    api.create_repo(repo_id, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=str(STAGE), repo_id=repo_id, commit_message="Upload LoRA adapter")
    print("uploaded:", f"https://huggingface.co/{repo_id}")